# Detector de spam por URL con un SVM

**Objetivo.** Predecir si una página web es spam usando solo su URL. El conjunto de datos es `url_spam.csv`, con la columna `url` (texto) y `is_spam` (verdadero o falso).

**Plan.** (1) Revisar los datos. (2) Quitar duplicados, partir en entrenamiento y prueba de forma estratificada y explorar las URL. (3) Preprocesar el texto (minúsculas, partir por signos de puntuación, quitar palabras vacías, lematizar). (4) Entrenar un SVM con parámetros por defecto. (5) Optimizarlo con búsqueda en rejilla y validación cruzada solo sobre entrenamiento. (6) Evaluar una sola vez en prueba y guardar el modelo.

## Bloque 1: carga y revisión de los datos

In [1]:
import re, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEMILLA = 42
datos = pd.read_csv("../url_spam.csv")

print("Filas y columnas:", datos.shape)
print(datos.dtypes, "\n")
print("Nulos por columna:\n", datos.isna().sum(), "\n")

conteo = datos["is_spam"].value_counts()
print(f"No spam: {conteo[False]}  |  Spam: {conteo[True]}  ({conteo[True] / len(datos):.1%} spam)")
print(f"Línea base (siempre «no spam»): {conteo[False] / len(datos):.1%}\n")

print("Filas duplicadas:", datos.duplicated().sum())
print("URL repetidas con etiqueta distinta:",
      int((datos.groupby("url")["is_spam"].nunique() > 1).sum()))

datos["largo"] = datos["url"].str.len()
print(datos.groupby("is_spam")["largo"].describe().round(1))
print("\nEjemplos:")
print(datos.sample(6, random_state=SEMILLA)[["url", "is_spam"]].to_string())

Filas y columnas: (2999, 2)
url         str
is_spam    bool
dtype: object 

Nulos por columna:
 url        0
is_spam    0
dtype: int64 

No spam: 2303  |  Spam: 696  (23.2% spam)
Línea base (siempre «no spam»): 76.8%

Filas duplicadas: 630
URL repetidas con etiqueta distinta: 0
          count  mean   std   min   25%   50%    75%    max
is_spam                                                    
False    2303.0  82.1  34.1  16.0  56.0  82.0  102.0  269.0
True      696.0  44.8  20.2  16.0  33.0  38.0   51.0  135.0

Ejemplos:
                                                                                                  url  is_spam
1376                                                       https://link.morningbrew.com/manage/5z8/oc     True
932                                    https://www.sciencedaily.com/releases/2019/02/190214093359.htm    False
144                                   https://www.ft.com/content/eae603a4-a369-4801-a4cc-06232898a34f    False
1752                      

## Bloque 2: quitar duplicados, partir los datos y explorar solo el entrenamiento

Las 630 filas duplicadas se quitan **antes** de partir, para que la misma URL no caiga en entrenamiento y en prueba. La partición es estratificada (misma proporción de spam en ambos conjuntos) y usa la semilla 42. La exploración posterior usa solo el entrenamiento: la prueba no se toca hasta la evaluación final.

In [2]:
from sklearn.model_selection import train_test_split

datos = datos.drop(columns="largo").drop_duplicates().reset_index(drop=True)
print("Filas tras quitar duplicados:", len(datos))
print(datos["is_spam"].value_counts(), "\n")

X_ent, X_pru, y_ent, y_pru = train_test_split(
    datos["url"], datos["is_spam"], test_size=0.2,
    stratify=datos["is_spam"], random_state=SEMILLA)
print("Entrenamiento:", len(X_ent), f"({y_ent.mean():.1%} spam)")
print("Prueba:", len(X_pru), f"({y_pru.mean():.1%} spam)")
print("URL en ambos conjuntos:", len(set(X_ent) & set(X_pru)))

# Exploración solo con entrenamiento (la prueba no se toca)
ent = pd.DataFrame({"url": X_ent, "is_spam": y_ent})
ent["https"] = ent["url"].str.startswith("https")
ent["barras"] = ent["url"].str.count("/")
ent["digitos"] = ent["url"].str.count(r"\d")
ent["dominio"] = ent["url"].str.extract(r"https?://(?:www\.)?([^/]+)")[0]
print(ent.groupby("is_spam")[["https", "barras", "digitos"]].mean().round(2), "\n")
for etiqueta in [True, False]:
    print("Dominios más frecuentes, is_spam =", etiqueta)
    print(ent[ent["is_spam"] == etiqueta]["dominio"].value_counts().head(5), "\n")

Filas tras quitar duplicados: 2369
is_spam
False    2125
True      244
Name: count, dtype: int64 

Entrenamiento: 1895 (10.3% spam)
Prueba: 474 (10.3% spam)
URL en ambos conjuntos: 0
         https  barras  digitos
is_spam                        
False     0.99    5.36     7.15
True      0.97    3.98     1.53 

Dominios más frecuentes, is_spam = True
dominio
numlock.substack.com    34
morningbrew.com         11
docs.google.com          9
creativemornings.com     6
austinkleon.com          5
Name: count, dtype: int64 

Dominios más frecuentes, is_spam = False
dominio
nytimes.com        83
morningbrew.com    72
youtube.com        71
reuters.com        48
cnn.com            44
Name: count, dtype: int64 



## Bloque 3: preprocesar el texto de las URL

La limpieza no aprende nada de los datos, así que se aplica por separado a entrenamiento y a prueba sin filtrar información. Orden: pasar a minúsculas primero, cortar en todo lo que no sea una letra, quitar palabras vacías (NLTK en inglés más `http`, `https` y `www`) y tokens de una letra, y lematizar con WordNet. Los dígitos actúan como separadores y se descartan, como en la lección, aunque antes vimos que distinguen las clases. El TF-IDF se ajustará más adelante solo con entrenamiento, dentro de un Pipeline.

In [3]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

palabras_vacias = set(stopwords.words("english")) | {"http", "https", "www"}
lematizador = WordNetLemmatizer()

def separar(url):
    # minúsculas PRIMERO, luego cortar en todo lo que no sea letra
    return [t for t in re.split(r"[^a-z]+", url.lower()) if t]

def limpiar(url):
    tokens = [t for t in separar(url) if t not in palabras_vacias and len(t) > 1]
    return " ".join(lematizador.lemmatize(t) for t in tokens)

# Verificación: separar no pierde ninguna letra (el fallo de la lección)
ok = all("".join(separar(u)) == re.sub(r"[^a-z]", "", u.lower()) for u in datos["url"])
print("Separar no pierde letras:", ok)

T_ent = X_ent.apply(limpiar)
T_pru = X_pru.apply(limpiar)
print("URL vacías tras limpiar (ent / pru):", (T_ent == "").sum(), (T_pru == "").sum())
print(pd.DataFrame({"original": X_ent.head(5), "limpia": T_ent.head(5)}).to_string())

vocabulario = pd.Series(" ".join(T_ent).split()).value_counts()
print("\nTokens distintos en entrenamiento:", len(vocabulario))
print(vocabulario.head(10))

Separar no pierde letras: True


URL vacías tras limpiar (ent / pru): 0 0
                                                     original                                     limpia
1144                    https://www.tiktok.com/@evanatoredits                   tiktok com evanatoredits
686   https://digg.com/video/german-shepherd-sneaks-into-pool  digg com video german shepherd sneak pool
696                              https://digg.com/advertising                       digg com advertising
2274       https://monocle.com/radio/shows/the-entrepreneurs/        monocle com radio show entrepreneur
2291                                    https://hyperapp.dev/                               hyperapp dev

Tokens distintos en entrenamiento: 4790
com            1661
html            242
news            218
u               194
story           156
article         144
coronavirus     139
org             115
morningbrew      86
nytimes          85
Name: count, dtype: int64


## Bloque 4: línea base con un SVM por defecto

Se evalúa con validación cruzada estratificada de 5 pliegues **solo sobre el entrenamiento**. El TF-IDF va dentro de un `Pipeline`, de modo que en cada pliegue se ajusta únicamente con los datos de ese pliegue. Se compara contra un modelo que siempre responde «no spam». Como solo el 10,3 % es spam, la exactitud sola engaña: importan la precisión, el recall y el F1 de la clase spam.

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
metricas = {
    "exactitud": "accuracy",
    "precision_spam": make_scorer(precision_score, zero_division=0),
    "recall_spam": make_scorer(recall_score, zero_division=0),
    "f1_spam": make_scorer(f1_score, zero_division=0),
}
modelos = {
    "Siempre «no spam»": Pipeline([("tfidf", TfidfVectorizer()),
                                   ("clf", DummyClassifier(strategy="most_frequent"))]),
    "SVM por defecto": Pipeline([("tfidf", TfidfVectorizer()),
                                 ("clf", SVC(random_state=SEMILLA))]),
}
filas = {}
for nombre, modelo in modelos.items():
    r = cross_validate(modelo, T_ent, y_ent, cv=cv, scoring=metricas)
    filas[nombre] = {m: f"{r['test_' + m].mean():.3f} ± {r['test_' + m].std():.3f}" for m in metricas}
print(pd.DataFrame(filas).T.to_string())

                       exactitud precision_spam    recall_spam        f1_spam
Siempre «no spam»  0.897 ± 0.000  0.000 ± 0.000  0.000 ± 0.000  0.000 ± 0.000
SVM por defecto    0.930 ± 0.012  0.871 ± 0.131  0.379 ± 0.071  0.527 ± 0.088


## Bloque 5: búsqueda en rejilla

Se prueban kernel (lineal y RBF), `C`, `gamma` (solo RBF) y `class_weight` (ninguno o balanceado, porque el spam es solo el 10 % y el SVM por defecto tenía un recall bajo). Se elige por el **F1 de la clase spam** con los mismos 5 pliegues y solo sobre el entrenamiento. Como se elige la mejor entre 32 combinaciones con la misma validación cruzada, esa cifra es algo optimista: la evaluación honesta es la de la prueba, que se usa una sola vez.

In [5]:
from sklearn.model_selection import GridSearchCV

rejilla = [
    {"clf__kernel": ["linear"], "clf__C": [0.1, 1, 10, 100],
     "clf__class_weight": [None, "balanced"]},
    {"clf__kernel": ["rbf"], "clf__C": [0.1, 1, 10, 100],
     "clf__gamma": ["scale", 0.1, 1], "clf__class_weight": [None, "balanced"]},
]
busqueda = GridSearchCV(modelos["SVM por defecto"], rejilla, cv=cv,
                        scoring=metricas, refit="f1_spam", n_jobs=-1)
busqueda.fit(T_ent, y_ent)

res = pd.DataFrame(busqueda.cv_results_).sort_values("rank_test_f1_spam").head(8)
cols = ["param_clf__kernel", "param_clf__C", "param_clf__gamma", "param_clf__class_weight",
        "mean_test_f1_spam", "std_test_f1_spam",
        "mean_test_precision_spam", "mean_test_recall_spam"]
print(res[cols].round(3).to_string(index=False))
print("\nMejores parámetros:", busqueda.best_params_)
print(f"F1 spam en validación cruzada: {busqueda.best_score_:.3f}")

param_clf__kernel  param_clf__C param_clf__gamma param_clf__class_weight  mean_test_f1_spam  std_test_f1_spam  mean_test_precision_spam  mean_test_recall_spam
              rbf           1.0              0.1                balanced              0.600             0.027                     0.524                  0.703
           linear           1.0              NaN                balanced              0.587             0.048                     0.494                  0.723
           linear          10.0              NaN                     NaN              0.585             0.038                     0.513                  0.682
              rbf          10.0              0.1                     NaN              0.582             0.037                     0.548                  0.621
              rbf          10.0              0.1                balanced              0.576             0.043                     0.490                  0.703
              rbf         100.0              0

## Bloque 6: evaluación única en la prueba

Es la primera vez que se usa el conjunto de prueba y no se toma ninguna decisión después de verlo. Se evalúa el SVM ajustado y, solo como comparación, el SVM por defecto y el modelo que siempre dice «no spam», todos entrenados con el entrenamiento. Con unas 49 URL spam en la prueba, el intervalo de confianza (bootstrap, 2.000 remuestreos, semilla 42) será ancho.

In [6]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def resumen(y, pred):
    return {"exactitud": accuracy_score(y, pred),
            "precision_spam": precision_score(y, pred, zero_division=0),
            "recall_spam": recall_score(y, pred, zero_division=0),
            "f1_spam": f1_score(y, pred, zero_division=0)}

pred = {
    "Siempre «no spam»": modelos["Siempre «no spam»"].fit(T_ent, y_ent).predict(T_pru),
    "SVM por defecto": modelos["SVM por defecto"].fit(T_ent, y_ent).predict(T_pru),
    "SVM ajustado": busqueda.best_estimator_.predict(T_pru),
}
print(pd.DataFrame({k: resumen(y_pru, p) for k, p in pred.items()}).T.round(3).to_string())

p = pred["SVM ajustado"]
print("\n" + classification_report(y_pru, p, target_names=["no spam", "spam"], digits=3))
print("Matriz de confusión (filas = real, columnas = predicho):")
print(pd.DataFrame(confusion_matrix(y_pru, p), index=["real no spam", "real spam"],
                   columns=["pred no spam", "pred spam"]).to_string())

# Intervalo de confianza al 95 % por bootstrap sobre la prueba
rng = np.random.default_rng(SEMILLA)
y_arr, p_arr = y_pru.to_numpy(), np.asarray(p)
f1s, accs = [], []
for _ in range(2000):
    i = rng.integers(0, len(y_arr), len(y_arr))
    f1s.append(f1_score(y_arr[i], p_arr[i], zero_division=0))
    accs.append(accuracy_score(y_arr[i], p_arr[i]))
print("\nIC 95 % F1 spam:", np.percentile(f1s, [2.5, 97.5]).round(3))
print("IC 95 % exactitud:", np.percentile(accs, [2.5, 97.5]).round(3))

# Ejemplos de errores
err = pd.DataFrame({"url": X_pru, "real_spam": y_pru, "pred_spam": p})
print("\nSpam no detectado (falsos negativos):")
print(err[err["real_spam"] & ~err["pred_spam"]]["url"].head(6).to_string())
print("\nLegítimas marcadas como spam (falsos positivos):")
print(err[~err["real_spam"] & err["pred_spam"]]["url"].head(6).to_string())

                   exactitud  precision_spam  recall_spam  f1_spam
Siempre «no spam»      0.897           0.000        0.000    0.000
SVM por defecto        0.941           0.889        0.490    0.632
SVM ajustado           0.907           0.537        0.735    0.621

              precision    recall  f1-score   support

     no spam      0.968     0.927     0.947       425
        spam      0.537     0.735     0.621        49

    accuracy                          0.907       474
   macro avg      0.753     0.831     0.784       474
weighted avg      0.924     0.907     0.913       474

Matriz de confusión (filas = real, columnas = predicho):
              pred no spam  pred spam
real no spam           394         31
real spam               13         36



IC 95 % F1 spam: [0.509 0.719]
IC 95 % exactitud: [0.878 0.932]

Spam no detectado (falsos negativos):
1480                                   https://trends.co/
2169       https://www.planbonestep.com/how-plan-b-works/
963                                     https://tim.blog/
1455    https://trends.co/weekly-lecture-exclusive-offer/
949                       https://go.tim.blog/10x-output/
103     https://www.morningbrew.com/daily/stories/2020...

Legítimas marcadas como spam (falsos positivos):
978                                   https://mowned.com/
2170                        https://www.planbonestep.com/
2026                https://weshouldgettogether.com/books
1849                https://www.creosotemaps.com/blm2020/
1657                  https://www.estherperel.com/podcast
1153    https://ownyourcontent.wordpress.com/2019/04/0...


## Bloque 7: guardar el modelo y comprobarlo

Se guarda el `Pipeline` completo (TF-IDF y SVM ya ajustados) con el kernel y la semilla en el nombre del archivo. La limpieza `limpiar` no va dentro del archivo, así que hay que aplicarla a cada URL antes de predecir. Después se recarga el archivo y se comprueba que las predicciones son idénticas.

In [7]:
import os, pickle

os.makedirs("../models", exist_ok=True)
ruta = "../models/svm_rbf_seed42.pkl"
with open(ruta, "wb") as f:
    pickle.dump(busqueda.best_estimator_, f)

with open(ruta, "rb") as f:
    cargado = pickle.load(f)
print("Predicciones idénticas tras recargar:", bool((cargado.predict(T_pru) == pred["SVM ajustado"]).all()))
print("Archivo:", ruta, f"({os.path.getsize(ruta) / 1024:.0f} KB)")

# Ejemplo de uso con una URL inventada (hay que aplicar `limpiar` antes)
ejemplo = "https://www.ejemplo.com/newsletter/weekly-digest/"
print(ejemplo, "->", "spam" if cargado.predict([limpiar(ejemplo)])[0] else "no spam")

Predicciones idénticas tras recargar: True
Archivo: ../models/svm_rbf_seed42.pkl (275 KB)
https://www.ejemplo.com/newsletter/weekly-digest/ -> spam


## Conclusiones

- **Datos.** El archivo trae 2.999 filas, 630 duplicadas exactas (452 de ellas spam). Tras deduplicar quedan 2.369 URL con un 10,3 % de spam, así que la línea base real es 89,7 % y no 76,8 %.
- **Correcciones respecto a la lección.** Se pasa a minúsculas antes de limpiar (la lección borraba las letras mayúsculas) y el TF-IDF se ajusta solo con el entrenamiento dentro de un `Pipeline` (la lección lo ajustaba con todos los datos).
- **Modelo.** SVM con kernel RBF, `C=1`, `gamma=0,1` y `class_weight="balanced"`, elegido por el F1 de la clase spam en validación cruzada (0,600 frente a 0,527 del SVM por defecto).
- **Resultado en la prueba** (474 URL, 49 spam). Exactitud 0,907 (IC 95 %: 0,878 a 0,932), precisión de spam 0,537, recall 0,735 y F1 0,621 (IC 95 %: 0,509 a 0,719). Detecta 36 de 49 spam con 31 falsas alarmas.
- **Comparación.** La exactitud apenas supera la línea base (0,897). El SVM por defecto da F1 0,632, equivalente. El ajuste desplazó el equilibrio hacia más recall a costa de precisión, pero no mejoró el modelo. No se cambió de modelo tras ver la prueba, para no contaminar la evaluación.
- **Límites.** La URL sola no separa todos los casos: `planbonestep.com` aparece como falso positivo y como falso negativo. Se descartan los dígitos aunque distinguían las clases, y la lematización convierte `us` en `u`. Con solo 49 URL spam en la prueba los intervalos son anchos.
- **Pasos siguientes.** Conseguir más ejemplos de spam, usar la estructura de la URL (dígitos, barras) como variables y mirar el contenido de la página.